# 创建两个DataLoader对象，分别用于加载训练集和测试集

In [1]:
# 运行准备：按示例代码5.1～5.3组织数据和fe，并使用示例代码5.43定义FeatSet
import os
import csv
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer

data_file = "../pybook-data/ch5/imdb_labelled.txt"
df = pd.read_csv(data_file, names=["sentence", "label"], sep="\t", quoting=csv.QUOTE_NONE)
sents = df["sentence"].values
y = df["label"].values

out_dir = "output"
train_file = f"{out_dir}/imdb_labelled_train.csv"
test_file = f"{out_dir}/imdb_labelled_test.csv"
if os.path.exists(train_file) and os.path.exists(test_file):
    print(f'Files exist at "{train_file}" and "{test_file}"')
    df_train, df_test = pd.read_csv(train_file), pd.read_csv(test_file)
    sents_train, sents_test = df_train["sentence"].values, df_test["sentence"].values
    y_train, y_test = df_train["label"].values, df_test["label"].values
else:
    os.makedirs(out_dir, exist_ok=True)
    sents_train, sents_test, y_train, y_test = train_test_split(sents, y, test_size=0.2, random_state=1)
    print(f'训练样本数量：{len(sents_train)}')
    print(f'测试样本数量：{len(sents_test)}')
    df_train = pd.DataFrame({'sentence': sents_train, 'label': y_train})
    df_test = pd.DataFrame({'sentence': sents_test, 'label': y_test})
    df_train.to_csv(train_file, index=False)
    df_test.to_csv(test_file, index=False)

fe = CountVectorizer()
fe.fit(sents_train)
vob = fe.get_feature_names_out()
print(f'词表大小：{len(vob)}')
x_train = fe.transform(sents_train)
x_test = fe.transform(sents_test)
print(f'训练集特征规模：{x_train.shape}')
print(f'测试集特征规模：{x_test.shape}')
import torch
from torch.utils.data import Dataset

class FeatSet(Dataset):
    def __init__(self, feats, labels):
        self.feats = feats
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        dense = self.feats[idx].toarray().squeeze()
        return torch.tensor(dense, dtype=torch.float32), torch.tensor(self.labels[idx], dtype=torch.long)

Files exist at "output/imdb_labelled_train.csv" and "output/imdb_labelled_test.csv"
词表大小：2647
训练集特征规模：(800, 2647)
测试集特征规模：(200, 2647)


In [2]:
from torch.utils.data import DataLoader

train_ds = FeatSet(fe.transform(sents_train), y_train)
test_ds = FeatSet(fe.transform(sents_test), [0]*len(sents_test)) #推理阶段无真实标签可用
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)
print(len(train_loader), len(test_loader)) # 13 7

13 7
